# Session 10: Secure Coding & OWASP Top 10

> Eliminate SQL Injection, XSS, and Broken Access Control. Zero-tolerance for hardcoded secrets.

## 1. SQL Injection & XSS Prevention

**SQL Injection** is the #1 web security vulnerability. It occurs when user-supplied input is concatenated directly into a SQL query string, allowing an attacker to alter the query's logic — extracting data, bypassing authentication, or deleting records.

**Why it's easy to introduce by accident:**
```python
# DANGEROUS — never do this
query = f"SELECT * FROM users WHERE username = '{username}'"
```
If `username` is `' OR '1'='1`, the query becomes `SELECT * FROM users WHERE username = '' OR '1'='1'` — which returns every row in the table.

**The fix is always parameterized queries (prepared statements).** The database driver handles quoting and escaping:
```python
# SAFE
cursor.execute("SELECT * FROM users WHERE username = %s", (username,))
```
When using an ORM like SQLAlchemy, always use the ORM's query API — never call `text()` or `execute()` with f-strings.

---

**Cross-Site Scripting (XSS)** is the equivalent vulnerability for HTML output. If user input is rendered into HTML without escaping, an attacker can inject `<script>` tags that run in another user's browser — stealing session cookies, redirecting to phishing sites, or keylogging.

**Prevention rules:**
1. **Always escape output** when rendering user-supplied data into HTML. Modern template engines (Jinja2, React, Angular) do this by default — never use `{{ value | safe }}` / `dangerouslySetInnerHTML` unless you have a strong reason.
2. Use a **Content Security Policy (CSP)** header to restrict which scripts the browser is allowed to run.
3. **Sanitize** rich-text input (e.g., blog post content) with a library like `bleach` that strips dangerous tags while keeping safe ones.

In [ ]:
# ❌ SQL Injection — NEVER do this
user_input = "' OR '1'='1"
bad_query = f"SELECT * FROM users WHERE email='{user_input}'"
print('VULNERABLE:', bad_query)

# ✅ Parameterized queries — always
def get_user_by_email(cursor, email: str):
    cursor.execute('SELECT * FROM users WHERE email = %s', (email,))
    return cursor.fetchone()

# ❌ XSS — rendering raw user input
bad_html = f'<div>{user_input}</div>'  # Executes scripts

# ✅ Always escape/sanitize
import html
safe_html = f'<div>{html.escape(user_input)}</div>'
print('Safe HTML:', safe_html)

## 2. Secrets Management

**A "secret" is any value that grants access to a protected resource:** API keys, database passwords, JWT signing keys, OAuth tokens, TLS private keys, encryption keys.

**The most common mistake** is storing secrets directly in source code or checked-in config files:
```python
# CATASTROPHIC — this ends up in git history forever
DB_PASSWORD = "super_secret_password"
API_KEY = "sk-prod-abc123xyz"
```
Once committed to a git repository, a secret is permanently compromised — even if you delete it in a later commit, it remains visible in the git history. Real-world breaches have been caused by exactly this mistake: a developer commits a credential, a bot scans GitHub, and within minutes the cloud account is drained.

**The two-layer approach:**

1. **Environment variables for local development:** Store secrets in a `.env` file (which must be in `.gitignore`) and load them with `python-dotenv`. The actual values are never in the repository.

2. **Secrets management services for production:** AWS Secrets Manager, HashiCorp Vault, GCP Secret Manager, and Azure Key Vault provide encrypted storage, fine-grained access control (only specific services can read specific secrets), automatic rotation, and full audit logs of every read.

**Rotation matters:** Secrets should be treated as temporary. A secrets manager that supports automatic rotation means a leaked key is useless after the rotation window.

**Principle of Least Privilege:** Each service should only have access to the secrets it actually needs. A background job that only sends emails should not have the database password.

In [ ]:
import os

# ❌ NEVER hardcode credentials
# API_KEY = 'sk-prod-abc123xyz'  # This gets committed to Git!
# DB_PASS = 'admin123'           # Exposed in code history forever

# ✅ Always use environment variables or secrets manager
API_KEY = os.getenv('OPENAI_API_KEY')  # From .env (gitignored)
DB_PASS = os.getenv('DB_PASSWORD')     # From vault/secrets manager

if not API_KEY:
    raise EnvironmentError('OPENAI_API_KEY not set — check .env or secrets vault')

print('Secrets loaded securely from environment')
print('Never log actual secret values — only confirm presence')

## 💡 Additional Examples: Secure Coding

**Example 1 — SQL Injection in an ORM**
Using an ORM doesn't automatically make you safe. Many developers call `session.execute(text(f"...{user_input}..."))` and unknowingly reintroduce the vulnerability. The safe practice is to use the ORM's query builder exclusively (`session.query(User).filter(User.email == email)`) and treat `text()` as a red flag in code review.

**Example 2 — Secrets rotation**
A secret that is never rotated is a ticking time bomb. If the secret leaked six months ago and you don't know it, an attacker may have had six months of silent access. Using a secrets manager with automated rotation means the window of exposure is limited to the rotation interval (e.g., 24 hours). The code retrieves the current secret by name at runtime — it never hardcodes a value.

**Example 3 — Input validation as a security boundary**
Validation is not just about user experience — it's a security control. Every input field that accepts a filename, URL, email, or command string is a potential injection point. Whitelisting (only allow known-good patterns via regex) is stronger than blacklisting (trying to block known-bad patterns). For example, a filename should only contain `[a-zA-Z0-9._-]` — anything else is rejected before it reaches the database or filesystem.

In [ ]:
# ─── Example 1: Password Hashing with PBKDF2 ─────────────────────────────────
import hashlib, hmac, secrets, base64, os

# ❌ NEVER store plain-text passwords
plain_password = 'MyP@ssw0rd!'

# ❌ NEVER use MD5/SHA1/SHA256 directly without a salt
bad_hash = hashlib.sha256(plain_password.encode()).hexdigest()
print(f'Bad hash (sha256, no salt): {bad_hash[:20]}...')
print(f'   ← Vulnerable to rainbow table attacks!\n')

# ✅ Use PBKDF2 (in production prefer bcrypt or argon2)
def hash_password(password: str) -> str:
    """Hash a password with PBKDF2-HMAC-SHA256 + a random salt."""
    salt = secrets.token_bytes(32)                      # 256-bit random salt
    iterations = 600_000                                # NIST recommendation 2024
    key = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, iterations)
    # Store: algorithm$iterations$b64salt$b64hash
    return f'pbkdf2_sha256${iterations}${base64.b64encode(salt).decode()}${base64.b64encode(key).decode()}'

def verify_password(password: str, stored_hash: str) -> bool:
    """Constant-time comparison to prevent timing attacks."""
    parts = stored_hash.split('$')
    if len(parts) != 4 or parts[0] != 'pbkdf2_sha256':
        return False
    _, iterations, b64_salt, b64_key = parts
    salt = base64.b64decode(b64_salt)
    stored_key = base64.b64decode(b64_key)
    test_key = hashlib.pbkdf2_hmac('sha256', password.encode('utf-8'), salt, int(iterations))
    return hmac.compare_digest(test_key, stored_key)  # ← Constant-time — no timing leak

hashed = hash_password(plain_password)
print(f'Stored hash: {hashed[:60]}...')
print(f'Verify correct:   {verify_password(plain_password, hashed)}')
print(f'Verify wrong:     {verify_password("WrongPassword", hashed)}')

# ─── Example 2: Secure JWT Token Creation and Validation ──────────────────────
import json, time

# Simplified JWT encode/decode — in production use PyJWT or python-jose
def base64url_encode(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b'=').decode()

def base64url_decode(s: str) -> bytes:
    padding = 4 - len(s) % 4
    return base64.urlsafe_b64decode(s + '=' * padding)

SECRET_KEY = os.getenv('JWT_SECRET', 'dev-secret-change-in-prod-min-256-bits')

def create_jwt(payload: dict, expiry_seconds: int = 3600) -> str:
    header = base64url_encode(json.dumps({'alg': 'HS256', 'typ': 'JWT'}).encode())
    payload_with_exp = {**payload, 'iat': int(time.time()), 'exp': int(time.time()) + expiry_seconds}
    body = base64url_encode(json.dumps(payload_with_exp).encode())
    signature_input = f'{header}.{body}'.encode()
    signature = base64url_encode(hmac.new(SECRET_KEY.encode(), signature_input, hashlib.sha256).digest())
    return f'{header}.{body}.{signature}'

def validate_jwt(token: str) -> dict:
    parts = token.split('.')
    if len(parts) != 3:
        raise ValueError('Invalid JWT structure')
    header, body, signature = parts
    # ✅ Verify signature FIRST — before reading any payload data
    signature_input = f'{header}.{body}'.encode()
    expected_sig = base64url_encode(hmac.new(SECRET_KEY.encode(), signature_input, hashlib.sha256).digest())
    if not hmac.compare_digest(signature, expected_sig):
        raise ValueError('Invalid JWT signature — possible tampering!')
    # ✅ Only then decode and validate claims
    payload = json.loads(base64url_decode(body))
    if payload.get('exp', 0) < time.time():
        raise ValueError('JWT token has expired')
    return payload

token = create_jwt({'user_id': 42, 'role': 'admin'})
print(f'\nJWT created: {token[:50]}...')
validated = validate_jwt(token)
print(f'Validated payload: user_id={validated["user_id"]}, role={validated["role"]}')

# Test tampered token detection
try:
    tampered = token[:-5] + 'XXXXX'
    validate_jwt(tampered)
except ValueError as e:
    print(f'Tampered token caught: {e}')

# ─── Example 3: Path Traversal Prevention ────────────────────────────────────
import re
from pathlib import Path

UPLOAD_DIR = '/var/app/uploads'  # Base directory — files must stay within this

# ❌ Path traversal vulnerability — attacker can escape the uploads directory
def download_file_bad(filename: str) -> str:
    path = f'/var/app/uploads/{filename}'
    return f'Reading: {path}'  # '../../etc/passwd' escapes uploads!

print(f'\n❌ Path traversal: {download_file_bad("../../etc/passwd")}')

# ✅ Safe file access using path canonicalization
def download_file_safe(filename: str) -> str:
    # Step 1: Strip all characters that are not word chars, dash, or dot
    safe_name = re.sub(r'[^\w\-.]', '_', filename)
    safe_name = safe_name.lstrip('.')  # Reject hidden files

    if not safe_name:
        raise ValueError('Invalid filename')

    # Step 2: Resolve the canonical absolute path and verify it stays inside UPLOAD_DIR
    base = Path(UPLOAD_DIR).resolve()
    target = (base / safe_name).resolve()

    # ✅ The critical security check — reject anything outside the base directory
    if not str(target).startswith(str(base)):
        raise PermissionError(f'Access denied: {filename} is outside uploads directory')

    return f'Reading: {target}'

safe_cases = ['report_2026.pdf', 'invoice-001.xlsx', 'logo.png']
attack_cases = ['../../etc/passwd', '../config/secrets.env', '%2e%2e/private']

print('\n✅ Safe filenames:')
for f in safe_cases:
    print(f'  {download_file_safe(f)}')

print('\n❌ Attack attempts (all blocked):')
for f in attack_cases:
    try:
        download_file_safe(f)
    except (PermissionError, ValueError) as e:
        print(f'  BLOCKED: {f!r} → {e}')


## 3. Broken Access Control

**OWASP #1 — Most Critical Vulnerability**

```python
# ❌ Direct object reference — any user can access any order
GET /api/orders/12345  # No ownership check!

# ✅ Enforce ownership + RBAC
def get_order(order_id: int, current_user: User) -> Order:
    order = db.get(order_id)
    if not order: raise NotFoundError('Order')
    if order.user_id != current_user.id and not current_user.is_admin:
        raise AuthorizationError()  # 403, not 404
    return order
```

## 4. AI Lab: Security Audit Simulation

### 🧪 AI Lab / Practice

> **TODO:** Select a controller/route handler from your project → Prompt Claude: 'Perform a security audit on this code. Check for: SQL injection, XSS, broken access control, hardcoded secrets, insecure direct object references. Provide severity rating (Critical/High/Medium) and fix for each finding.' → Fix all Critical and High findings.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')